# CNN Preprocessing Pipeline

Quality analysis (blank, dark, blurry detection), brightness profiling, deskewing,
and resizing to 224x224 for MobileNetV2.

Decisions based on dataset analysis:
- Blanks are kept (blankness discriminates FileFolder)
- Dark images are kept (dark covers have visual features)
- Blurry images are removed (no useful signal)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import defaultdict, Counter

from deskew import determine_skew

plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# paths
INPUT_DIR  = Path('rvlcdip_split')
OUTPUT_DIR = Path('CNN_preprocessed')

SPLITS = ['train', 'val', 'test']

CLASSES = [
    'Advertisement',
    'Budget',
    'Email',
    'FileFolder',
    'Form',
    'Handwritten',
    'Invoice',
    'Letter',
    'Memo',
    'News',
    'Presentation',
    'Questionnaire',
    'Resume',
    'ScientificPub',
    'ScientificReport',
    'Specification',
]

NUM_CLASSES = len(CLASSES)

# quality thresholds
BLANK_THRESHOLD = 0.997   # white pixel ratio above this = blank
DARK_THRESHOLD = 0.6      # dark pixel ratio above this = dark
BLURRY_THRESHOLD = 50.0   # Laplacian variance below this = blurry

# filtering decisions
REMOVE_BLANKS = False      # blankness discriminates FileFolder (72% of blanks)
REMOVE_BLURRY = True       # no useful visual or text signal

# image size for MobileNetV2
IMG_SIZE = 224

print(f"Classes: {NUM_CLASSES}")
print(f"Blank threshold: {BLANK_THRESHOLD}")
print(f"Dark threshold:  {DARK_THRESHOLD}")
print(f"Blurry threshold: {BLURRY_THRESHOLD}")
print(f"Remove blanks: {REMOVE_BLANKS}")
print(f"Remove blurry: {REMOVE_BLURRY}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")

In [ ]:
# verify train/val/test splits with class folders
print("Verifying split structure...")
print(f"Input directory: {INPUT_DIR.resolve()}")
print()

all_ok = True
for split in SPLITS:
    print(f"  {split}/")
    for cls in CLASSES:
        cls_dir = INPUT_DIR / split / cls
        if not cls_dir.exists():
            print(f"    NOT FOUND: {cls}")
            all_ok = False
            continue
        count = len([f for f in os.listdir(cls_dir)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))])
        print(f"    {cls:<22} {count:>5} images")
    print()

if all_ok:
    total = sum(
        len([f for f in os.listdir(INPUT_DIR / s / c)
             if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))])
        for s in SPLITS for c in CLASSES if (INPUT_DIR / s / c).exists()
    )
    print(f"All splits found. Total images: {total}")
else:
    print("Some folders missing. Check paths.")

In [ ]:
def is_blank(img_path, threshold=BLANK_THRESHOLD):
    """Check if an image is mostly blank (white).
    Returns (is_blank: bool, white_ratio: float)."""
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return True, 1.0
    white_ratio = np.sum(img > 240) / img.size
    return white_ratio >= threshold, white_ratio


# test on one image per class from train
print("Testing is_blank on one train sample per class:")
print("-" * 52)
for cls in CLASSES:
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    sample = sorted(os.listdir(cls_dir))[0]
    path = cls_dir / sample
    blank, ratio = is_blank(path)
    status = "BLANK" if blank else "ok"
    print(f"  {cls:<22} white_ratio={ratio:.4f}  {status}")

In [ ]:
# show white ratios for 5 samples from 4 classes
sample_classes = CLASSES[:4]

fig, axes = plt.subplots(4, 5, figsize=(15, 10))
fig.suptitle("White Ratio Check (train samples)", fontsize=14)

for row, cls in enumerate(sample_classes):
    cls_dir = INPUT_DIR / 'train' / cls
    files = sorted(os.listdir(cls_dir))[:5]
    for col, fname in enumerate(files):
        ax = axes[row][col]
        img = Image.open(cls_dir / fname)
        ax.imshow(img, cmap='gray')
        _, ratio = is_blank(cls_dir / fname)
        color = 'red' if ratio >= BLANK_THRESHOLD else 'black'
        ax.set_title(f"{ratio:.3f}", fontsize=9, color=color)
        ax.axis('off')
    axes[row][0].set_ylabel(cls, fontsize=10, rotation=0, labelpad=60)

plt.tight_layout()
plt.show()

In [ ]:
# scan every image across all splits for blanks
print("Scanning for blank images...")
blank_records = {}  # (split, class) -> list of blank filenames

for split in SPLITS:
    for cls in CLASSES:
        cls_dir = INPUT_DIR / split / cls
        if not cls_dir.exists():
            blank_records[(split, cls)] = []
            continue
        blanks = []
        files = sorted(os.listdir(cls_dir))
        for fname in files:
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
                continue
            blank, _ = is_blank(cls_dir / fname)
            if blank:
                blanks.append(fname)
        blank_records[(split, cls)] = blanks

total_blanks = sum(len(v) for v in blank_records.values())
print(f"Blank images found: {total_blanks}")
print()
for split in SPLITS:
    print(f"  {split}:")
    for cls in CLASSES:
        n = len(blank_records.get((split, cls), []))
        if n > 0:
            print(f"    {cls:<22} {n:>3} blanks")
    print()

In [ ]:
# visualize blank images with class and white ratio (report quality)
has_blanks = [(k, recs) for k, recs in blank_records.items() if recs]

if has_blanks:
    blank_info = []
    for (split, cls), recs in has_blanks:
        for fname in recs:
            path = INPUT_DIR / split / cls / fname
            _, ratio = is_blank(path)
            blank_info.append((path, split, cls, fname, ratio))

    n = len(blank_info)
    cols = min(n, 5)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.2), dpi=200)
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    fig.suptitle("Blank Images Detected (white ratio >= {:.3f})".format(BLANK_THRESHOLD),
                 fontsize=14, fontweight='bold', y=1.01)

    for idx, (path, split, cls, fname, ratio) in enumerate(blank_info):
        r, c = divmod(idx, cols)
        ax = axes[r][c] if rows > 1 else axes[c]
        img = Image.open(path)
        ax.imshow(img, cmap='gray')
        ax.set_title(f"{cls}\nwhite ratio = {ratio:.4f}", fontsize=9, pad=6)
        ax.set_xlabel(f"{split}/{fname[:20]}", fontsize=7, color='gray')
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for idx in range(n, rows * cols):
        r, c = divmod(idx, cols)
        ax = axes[r][c] if rows > 1 else axes[c]
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('blank_images_report.png', dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()
    print(f"Saved to blank_images_report.png ({n} blank images)")
else:
    print("No blank images found.")

In [ ]:
def is_dark(img_path, threshold=DARK_THRESHOLD):
    """Check if an image is mostly dark (black).
    Returns (is_dark: bool, dark_ratio: float)."""
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False, 0.0
    dark_ratio = np.sum(img < 15) / img.size
    return dark_ratio >= threshold, dark_ratio


def get_brightness_stats(img_path):
    """Returns (white_ratio, dark_ratio, mean_brightness)."""
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 1.0, 0.0, 255.0
    white_ratio = np.sum(img > 240) / img.size
    dark_ratio = np.sum(img < 15) / img.size
    mean_brightness = np.mean(img)
    return white_ratio, dark_ratio, mean_brightness


# test on one image per class from train
print("Testing brightness stats on one train sample per class:")
print("-" * 62)
for cls in CLASSES:
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    sample = sorted(os.listdir(cls_dir))[0]
    path = cls_dir / sample
    dark, dr = is_dark(path)
    blank, wr = is_blank(path)
    status = "DARK" if dark else ("BLANK" if blank else "ok")
    print(f"  {cls:<22} white={wr:.3f}  dark={dr:.3f}  {status}")

In [ ]:
# scan every image across all splits for dark
print("Scanning for dark images...")
dark_records = {}

for split in SPLITS:
    for cls in CLASSES:
        cls_dir = INPUT_DIR / split / cls
        if not cls_dir.exists():
            dark_records[(split, cls)] = []
            continue
        darks = []
        files = sorted(os.listdir(cls_dir))
        for fname in files:
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
                continue
            dark, _ = is_dark(cls_dir / fname)
            if dark:
                darks.append(fname)
        dark_records[(split, cls)] = darks

total_dark = sum(len(v) for v in dark_records.values())
print(f"Dark images found: {total_dark}")
print()
for split in SPLITS:
    print(f"  {split}:")
    for cls in CLASSES:
        n = len(dark_records.get((split, cls), []))
        if n > 0:
            print(f"    {cls:<22} {n:>3} dark")
    print()

In [ ]:
# measure white_ratio and dark_ratio for every train image per class
print("Computing brightness stats for all train images...")
class_stats = {}  # cls -> list of (white_ratio, dark_ratio, mean_brightness)

for cls in CLASSES:
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        class_stats[cls] = []
        continue
    stats = []
    files = sorted(os.listdir(cls_dir))
    for fname in files:
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
            continue
        path = cls_dir / fname
        wr, dr, mb = get_brightness_stats(path)
        stats.append((wr, dr, mb))
    class_stats[cls] = stats

total = sum(len(v) for v in class_stats.values())
print(f"Done. {total} images scanned across {NUM_CLASSES} classes.")

In [ ]:
# summary: per class, how many blank / dark / normal
print(f"{'Class':<22} {'Total':>5} {'Blank':>5} {'Dark':>5} {'Normal':>6} {'Avg White':>9} {'Avg Dark':>9}")
print("-" * 72)

for cls in CLASSES:
    stats = class_stats.get(cls, [])
    n_total = len(stats)
    n_blank = sum(1 for wr, dr, mb in stats if wr >= BLANK_THRESHOLD)
    n_dark = sum(1 for wr, dr, mb in stats if dr >= DARK_THRESHOLD)
    n_normal = n_total - n_blank - n_dark
    avg_wr = np.mean([wr for wr, dr, mb in stats]) if stats else 0
    avg_dr = np.mean([dr for wr, dr, mb in stats]) if stats else 0
    print(f"  {cls:<20} {n_total:>5} {n_blank:>5} {n_dark:>5} {n_normal:>6} {avg_wr:>9.3f} {avg_dr:>9.3f}")

print("-" * 72)

In [ ]:
# scatter plot: white_ratio vs dark_ratio per class
fig, ax = plt.subplots(figsize=(10, 7), dpi=150)

for cls in CLASSES:
    pts = [(d[0], d[1]) for d in class_stats.get(cls, [])]
    if not pts:
        continue
    xs, ys = zip(*pts)
    if cls == 'FileFolder':
        ax.scatter(xs, ys, s=12, alpha=0.7, label=cls, color='red', zorder=5)
    else:
        ax.scatter(xs, ys, s=6, alpha=0.2, label=cls)

ax.set_xlabel("White Ratio (pixels > 240)", fontsize=11)
ax.set_ylabel("Dark Ratio (pixels < 15)", fontsize=11)
ax.set_title("White vs Dark Ratio per Image (train)", fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=7, markerscale=2)
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.3)
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.3)
plt.tight_layout()
plt.savefig('white_vs_dark_scatter.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# white_ratio histogram per class (4x4 grid)
fig, axes = plt.subplots(4, 4, figsize=(16, 12), dpi=150)
fig.suptitle("White Ratio Distribution per Class (train)", fontsize=14, fontweight='bold')

for idx, cls in enumerate(CLASSES):
    row, col = divmod(idx, 4)
    ax = axes[row][col]
    stats = class_stats.get(cls, [])
    ratios = [wr for wr, dr, mb in stats]
    ax.hist(ratios, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(BLANK_THRESHOLD, color='red', linestyle='--', linewidth=1)
    ax.set_title(cls, fontsize=9, fontweight='bold')
    ax.set_xlim(0, 1)
    if col == 0:
        ax.set_ylabel("Count", fontsize=8)
    if row == 3:
        ax.set_xlabel("White Ratio", fontsize=8)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('white_ratio_per_class.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# dark_ratio histogram per class (4x4 grid)
fig, axes = plt.subplots(4, 4, figsize=(16, 12), dpi=150)
fig.suptitle("Dark Ratio Distribution per Class (train)", fontsize=14, fontweight='bold')

for idx, cls in enumerate(CLASSES):
    row, col = divmod(idx, 4)
    ax = axes[row][col]
    stats = class_stats.get(cls, [])
    ratios = [dr for wr, dr, mb in stats]
    ax.hist(ratios, bins=40, color='dimgray', edgecolor='white', alpha=0.8)
    ax.axvline(DARK_THRESHOLD, color='red', linestyle='--', linewidth=1)
    ax.set_title(cls, fontsize=9, fontweight='bold')
    ax.set_xlim(0, 1)
    if col == 0:
        ax.set_ylabel("Count", fontsize=8)
    if row == 3:
        ax.set_xlabel("Dark Ratio", fontsize=8)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('dark_ratio_per_class.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# show darkest and brightest sample per class
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(NUM_CLASSES * 2, 5), dpi=200)
fig.suptitle("Darkest (top) vs Brightest (bottom) Sample per Class",
             fontsize=13, fontweight='bold')

for idx, cls in enumerate(CLASSES):
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        axes[0][idx].axis('off')
        axes[1][idx].axis('off')
        continue
    stats = class_stats.get(cls, [])
    files = sorted([f for f in os.listdir(cls_dir)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))])

    indexed = [(mb, fname) for (wr, dr, mb), fname in zip(stats, files)]
    indexed.sort(key=lambda x: x[0])

    # darkest
    darkest_fname = indexed[0][1]
    darkest_mb = indexed[0][0]
    img_dark = Image.open(cls_dir / darkest_fname)
    axes[0][idx].imshow(img_dark, cmap='gray')
    axes[0][idx].set_title(cls, fontsize=7, fontweight='bold')
    axes[0][idx].set_xlabel(f"mean={darkest_mb:.0f}", fontsize=6)
    axes[0][idx].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # brightest
    brightest_fname = indexed[-1][1]
    brightest_mb = indexed[-1][0]
    img_bright = Image.open(cls_dir / brightest_fname)
    axes[1][idx].imshow(img_bright, cmap='gray')
    axes[1][idx].set_xlabel(f"mean={brightest_mb:.0f}", fontsize=6)
    axes[1][idx].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

axes[0][0].set_ylabel("Darkest", fontsize=9, rotation=0, labelpad=50)
axes[1][0].set_ylabel("Brightest", fontsize=9, rotation=0, labelpad=50)
plt.tight_layout()
plt.savefig('darkest_brightest_per_class.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# show FileFolder samples across the brightness spectrum
ff_data = [(d[0], d[1], d[2]) for d in class_stats.get('FileFolder', [])]
ff_data.sort(key=lambda x: x[0])  # sort by white_ratio

n_show = 5
dark_picks = ff_data[:n_show]
bright_picks = ff_data[-n_show:]

ff_dir = INPUT_DIR / 'train' / 'FileFolder'
ff_files = sorted(os.listdir(ff_dir))

fig, axes = plt.subplots(2, n_show, figsize=(n_show * 3, 6), dpi=200)
fig.suptitle("FileFolder: Dark (top) vs Bright (bottom) Samples", fontsize=13, fontweight='bold')

for col, (wr, dr, mb) in enumerate(dark_picks):
    for fname in ff_files:
        path = ff_dir / fname
        w, d, m = get_brightness_stats(path)
        if abs(w - wr) < 0.001 and abs(d - dr) < 0.001:
            img = Image.open(path)
            axes[0][col].imshow(img, cmap='gray')
            axes[0][col].set_title(f"white={wr:.3f}\ndark={dr:.3f}", fontsize=8)
            axes[0][col].axis('off')
            break

for col, (wr, dr, mb) in enumerate(bright_picks):
    for fname in ff_files:
        path = ff_dir / fname
        w, d, m = get_brightness_stats(path)
        if abs(w - wr) < 0.001 and abs(d - dr) < 0.001:
            img = Image.open(path)
            axes[1][col].imshow(img, cmap='gray')
            axes[1][col].set_title(f"white={wr:.3f}\ndark={dr:.3f}", fontsize=8)
            axes[1][col].axis('off')
            break

axes[0][0].set_ylabel("Dark\n(black covers)", fontsize=10, rotation=0, labelpad=60)
axes[1][0].set_ylabel("Bright\n(blank pages)", fontsize=10, rotation=0, labelpad=60)
plt.tight_layout()
plt.savefig('filefolder_bimodal.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# check which classes have bimodal brightness
print("Bimodality check per class:")
print(f"{'Class':<22} {'Blank':>5} {'Dark':>5} {'Bimodal?':>8}  {'Fusion Recommendation'}")
print("-" * 80)

for cls in CLASSES:
    stats = class_stats.get(cls, [])
    n_blank = sum(1 for wr, dr, mb in stats if wr >= BLANK_THRESHOLD)
    n_dark = sum(1 for wr, dr, mb in stats if dr >= DARK_THRESHOLD)
    bimodal = n_blank > 5 and n_dark > 5

    if bimodal:
        rec = "CNN-heavy + brightness rule (bimodal hurts OCR)"
    elif n_blank > 5:
        rec = "CNN-heavy fusion (blank pages hurt OCR)"
    elif n_dark > 5:
        rec = "CNN-heavy fusion (dark backgrounds hurt OCR)"
    else:
        rec = "Balanced or OCR-heavy fusion"

    bimodal_str = "YES" if bimodal else "no"
    print(f"  {cls:<20} {n_blank:>5} {n_dark:>5} {bimodal_str:>8}  {rec}")

In [ ]:
def is_blurry(img_path, threshold=BLURRY_THRESHOLD):
    """Check if an image is blurry using Laplacian variance.
    Returns (is_blurry: bool, score: float)."""
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return True, 0.0
    score = cv2.Laplacian(img, cv2.CV_64F).var()
    return score < threshold, score


# test on one image per class from train
print("Testing is_blurry on one train sample per class:")
print("-" * 52)
for cls in CLASSES:
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    sample = sorted(os.listdir(cls_dir))[0]
    path = cls_dir / sample
    blurry, score = is_blurry(path)
    status = "BLURRY" if blurry else "ok"
    print(f"  {cls:<22} laplacian_var={score:.1f}  {status}")

In [ ]:
# show blur scores for 5 samples from 4 classes
sample_classes = CLASSES[:4]

fig, axes = plt.subplots(4, 5, figsize=(15, 10))
fig.suptitle("Blur Score Check (train samples)", fontsize=14)

for row, cls in enumerate(sample_classes):
    cls_dir = INPUT_DIR / 'train' / cls
    files = sorted(os.listdir(cls_dir))[:5]
    for col, fname in enumerate(files):
        ax = axes[row][col]
        img = Image.open(cls_dir / fname)
        ax.imshow(img, cmap='gray')
        _, score = is_blurry(cls_dir / fname)
        color = 'red' if score < BLURRY_THRESHOLD else 'black'
        ax.set_title(f"{score:.0f}", fontsize=9, color=color)
        ax.axis('off')
    axes[row][0].set_ylabel(cls, fontsize=10, rotation=0, labelpad=60)

plt.tight_layout()
plt.show()

In [ ]:
# scan every image for blur across all splits
print("Scanning for blurry images...")
blurry_records = {}

for split in SPLITS:
    for cls in CLASSES:
        cls_dir = INPUT_DIR / split / cls
        if not cls_dir.exists():
            blurry_records[(split, cls)] = []
            continue
        blurrys = []
        files = sorted(os.listdir(cls_dir))
        for fname in files:
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
                continue
            blurry, _ = is_blurry(cls_dir / fname)
            if blurry:
                blurrys.append(fname)
        blurry_records[(split, cls)] = blurrys

total_blurry = sum(len(v) for v in blurry_records.values())
print(f"Blurry images found: {total_blurry}")
print()
for split in SPLITS:
    print(f"  {split}:")
    for cls in CLASSES:
        n = len(blurry_records.get((split, cls), []))
        if n > 0:
            print(f"    {cls:<22} {n:>3} blurry")
    print()

In [ ]:
# histogram of blur scores across train images
all_scores = []

for cls in CLASSES:
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    files = sorted(os.listdir(cls_dir))
    for fname in files:
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
            continue
        _, score = is_blurry(cls_dir / fname)
        all_scores.append(score)

plt.figure(figsize=(10, 4))
plt.hist(all_scores, bins=80, color='steelblue', edgecolor='white')
plt.axvline(BLURRY_THRESHOLD, color='red', linestyle='--', linewidth=1.5,
            label=f'Threshold={BLURRY_THRESHOLD}')
plt.xlabel("Laplacian Variance (blur score)")
plt.ylabel("Number of images")
plt.title("Blur Score Distribution (train split)")
plt.legend(loc='best')
plt.tight_layout()
plt.savefig('blur_score_histogram.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
def detect_skew_angle(img_path):
    """Detect skew angle in degrees using the deskew library."""
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.0
    angle = determine_skew(img)
    return angle if angle is not None else 0.0


def fix_skew(img_array):
    """Rotate image to correct skew. Returns (corrected_array, angle)."""
    gray = img_array if len(img_array.shape) == 2 else cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
    angle = determine_skew(gray)
    if angle is None or abs(angle) < 0.5:
        return img_array, 0.0
    h, w = img_array.shape[:2]
    center = (w // 2, h // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(img_array, matrix, (w, h),
                             flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


# test on one image per class from train
print("Testing skew detection on train samples:")
print("-" * 52)
for cls in CLASSES[:8]:
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    sample = sorted(os.listdir(cls_dir))[0]
    path = cls_dir / sample
    angle = detect_skew_angle(path)
    print(f"  {cls:<22} skew: {angle:+.2f} deg")

In [ ]:
# before/after deskew for 8 classes
fig, axes = plt.subplots(2, 8, figsize=(20, 5), dpi=150)
fig.suptitle("Deskew Before (top) / After (bottom)", fontsize=14)

for col, cls in enumerate(CLASSES[:8]):
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    sample = sorted(os.listdir(cls_dir))[0]
    img = cv2.imread(str(cls_dir / sample))

    axes[0][col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0][col].set_title(cls, fontsize=8)
    axes[0][col].axis('off')

    corrected, angle = fix_skew(img)
    axes[1][col].imshow(cv2.cvtColor(corrected, cv2.COLOR_BGR2RGB))
    axes[1][col].set_title(f"{angle:+.1f} deg", fontsize=8)
    axes[1][col].axis('off')

axes[0][0].set_ylabel("Before", fontsize=11)
axes[1][0].set_ylabel("After", fontsize=11)
plt.tight_layout()
plt.savefig('deskew_before_after.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
def preprocess_cnn(img_path, output_path, img_size=IMG_SIZE):
    """Full CNN preprocessing:
    1. Read image
    2. Deskew
    3. Resize to img_size x img_size
    4. Save to output_path
    Returns True on success, False on failure.
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return False

    # deskew
    img, angle = fix_skew(img)

    # resize
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)

    # save
    os.makedirs(os.path.dirname(str(output_path)), exist_ok=True)
    cv2.imwrite(str(output_path), img)
    return True


# test on one image
test_cls = CLASSES[0]
test_dir = INPUT_DIR / 'train' / test_cls
test_file = sorted(os.listdir(test_dir))[0]
test_in = test_dir / test_file
test_out = Path("test_output.png")

success = preprocess_cnn(test_in, test_out)
if success:
    img = Image.open(test_out)
    print(f"Preprocess test: {img.size} OK")
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Preprocessed ({test_cls})")
    plt.show()
    test_out.unlink()
else:
    print("Preprocess test FAILED")

In [ ]:
# show one preprocessed sample per class
fig, axes = plt.subplots(2, 8, figsize=(20, 5), dpi=150)
fig.suptitle("Preprocessed Samples (one per class, train)", fontsize=14)

for idx, cls in enumerate(CLASSES):
    row, col = divmod(idx, 8)
    ax = axes[row][col]
    cls_dir = INPUT_DIR / 'train' / cls
    if not cls_dir.exists():
        ax.axis('off')
        continue
    sample = sorted(os.listdir(cls_dir))[0]
    path = cls_dir / sample

    img = cv2.imread(str(path))
    img_corrected, angle = fix_skew(img)
    img_resized = cv2.resize(img_corrected, (IMG_SIZE, IMG_SIZE))

    ax.imshow(cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB))
    ax.set_title(cls, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('preprocessed_samples.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# process all images across train/val/test
total_ok = 0
total_fail = 0
total_skipped = 0

for split in SPLITS:
    print(f"Processing {split}...")
    split_ok = 0
    split_skip = 0
    split_fail = 0

    for cls in CLASSES:
        cls_dir = INPUT_DIR / split / cls
        if not cls_dir.exists():
            continue
        files = sorted(os.listdir(cls_dir))
        for fname in files:
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
                continue

            # skip blanks only if REMOVE_BLANKS is True
            if REMOVE_BLANKS and fname in blank_records.get((split, cls), []):
                split_skip += 1
                continue

            # skip blurry images
            if REMOVE_BLURRY and fname in blurry_records.get((split, cls), []):
                split_skip += 1
                continue

            in_path = cls_dir / fname
            out_path = OUTPUT_DIR / split / cls / fname
            ok = preprocess_cnn(in_path, out_path)

            if ok:
                split_ok += 1
            else:
                split_fail += 1

    print(f"  {split}: {split_ok} ok, {split_skip} skipped, {split_fail} failed")
    total_ok += split_ok
    total_skipped += split_skip
    total_fail += split_fail

print()
print(f"Total: {total_ok} processed, {total_skipped} skipped, {total_fail} failed")

In [ ]:
# verify output structure and counts
print("Output directory:", OUTPUT_DIR.resolve())
print()
print(f"{'Class':<22} {'Train':>5} {'Val':>6} {'Test':>6} {'Total':>6}")
print("-" * 48)

totals = {'train': 0, 'val': 0, 'test': 0}
for cls in CLASSES:
    counts = {}
    for split in SPLITS:
        d = OUTPUT_DIR / split / cls
        n = len(os.listdir(d)) if d.exists() else 0
        counts[split] = n
        totals[split] += n
    total = sum(counts.values())
    print(f"  {cls:<20} {counts['train']:>5} {counts['val']:>6} {counts['test']:>6} {total:>6}")

print("-" * 48)
grand = sum(totals.values())
print(f"  {'TOTAL':<18} {totals['train']:>5} {totals['val']:>6} {totals['test']:>6} {grand:>6}")

In [ ]:
# show one preprocessed image per class from output
fig, axes = plt.subplots(2, 8, figsize=(20, 5), dpi=150)
fig.suptitle("Final Output Samples (train, one per class)", fontsize=14)

for idx, cls in enumerate(CLASSES):
    row, col = divmod(idx, 8)
    ax = axes[row][col]
    cls_dir = OUTPUT_DIR / 'train' / cls
    if not cls_dir.exists() or not os.listdir(cls_dir):
        ax.axis('off')
        continue
    sample = sorted(os.listdir(cls_dir))[0]
    img = Image.open(cls_dir / sample)
    ax.imshow(img)
    ax.set_title(cls, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('final_output_samples.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# bar chart: blank + dark + blurry stats per class (train only)
blank_per_cls = [sum(len(blank_records.get((s, cls), [])) for s in SPLITS) for cls in CLASSES]
dark_per_cls = [sum(len(dark_records.get((s, cls), [])) for s in SPLITS) for cls in CLASSES]
blurry_per_cls = [sum(len(blurry_records.get((s, cls), [])) for s in SPLITS) for cls in CLASSES]

x = np.arange(NUM_CLASSES)
width = 0.25

plt.figure(figsize=(14, 5), dpi=150)
plt.bar(x - width, blank_per_cls, width, label='Blank', color='salmon')
plt.bar(x, dark_per_cls, width, label='Dark', color='dimgray')
plt.bar(x + width, blurry_per_cls, width, label='Blurry', color='khaki')
plt.xticks(x, CLASSES, rotation=45, ha='right', fontsize=8)
plt.ylabel("Images")
plt.title("Quality Analysis: Blank, Dark, and Blurry per Class")
plt.legend(loc='best')
plt.tight_layout()
plt.savefig('quality_analysis_bar.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# save preprocessing log
log_path = OUTPUT_DIR / 'preprocessing_log.txt'
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(log_path, 'w') as f:
    f.write("CNN Preprocessing Log\n")
    f.write("=" * 40 + "\n\n")
    f.write(f"Input directory:  {INPUT_DIR.resolve()}\n")
    f.write(f"Output directory: {OUTPUT_DIR.resolve()}\n\n")
    f.write(f"Blank threshold:  {BLANK_THRESHOLD}\n")
    f.write(f"Dark threshold:   {DARK_THRESHOLD}\n")
    f.write(f"Blurry threshold: {BLURRY_THRESHOLD}\n")
    f.write(f"Image size:       {IMG_SIZE}x{IMG_SIZE}\n")
    f.write(f"Remove blanks:    {REMOVE_BLANKS}\n")
    f.write(f"Remove blurry:    {REMOVE_BLURRY}\n\n")

    f.write("Per-class quality stats (all splits):\n")
    f.write(f"{'Class':<22} {'Blank':>5} {'Dark':>5} {'Blurry':>6}\n")
    f.write("-" * 40 + "\n")
    for i, cls in enumerate(CLASSES):
        f.write(f"  {cls:<20} {blank_per_cls[i]:>5} {dark_per_cls[i]:>5} {blurry_per_cls[i]:>6}\n")

    f.write(f"\nTotal blank:   {sum(blank_per_cls)}\n")
    f.write(f"Total dark:    {sum(dark_per_cls)}\n")
    f.write(f"Total blurry:  {sum(blurry_per_cls)}\n")
    f.write(f"Total processed: {total_ok}\n")
    f.write(f"Total skipped:   {total_skipped}\n")
    f.write(f"Total failed:    {total_fail}\n\n")

    f.write("Bimodality analysis:\n")
    for cls in CLASSES:
        stats = class_stats.get(cls, [])
        n_blank = sum(1 for wr, dr, mb in stats if wr >= BLANK_THRESHOLD)
        n_dark = sum(1 for wr, dr, mb in stats if dr >= DARK_THRESHOLD)
        bimodal = "YES" if (n_blank > 5 and n_dark > 5) else "no"
        f.write(f"  {cls:<22} blank={n_blank:>3}  dark={n_dark:>3}  bimodal={bimodal}\n")

print(f"Log saved to {log_path}")